# 🏛️ Calc Groups Cathedral — Quest Brief

> 🏗️ Build: **2026-05-29 12:41:15** &nbsp;·&nbsp; if you don't see this stamp after re-upload, close the notebook tab and reopen it.

> _An apprentice writes a measure for every KPI variation._
> _A master architect carves them all from **one stone**._

---

## 📜 Your mission

The CFO just asked for a **complete sales dashboard**. They want **12 KPIs** — each a
variation of the same `Sales Amount` base measure: this year, last year, YoY, YTD,
rolling 12 months, % of year, distinct customers… the full Time Intelligence catalogue.

Your job: open the **`Cathedral_Model`** semantic model in the workspace and **carve
12 DAX measures**, one per pillar. The names, the data, the test contexts — they're
all locked down. You just write the DAX.

When you're done, open the **`CalcGroups_Check`** notebook → run it → it grades each
measure, scores your DAX for **elegance**, and assigns you an **Architect rank**.

---

## 🛠️ How to add measures in the web modeler

1. In the workspace, open **`Cathedral_Model`** (semantic model).
2. Click **`Open data model`** → you'll see the diagram (Date, Customer, Sales, Budget).
3. Right-click on the **`Sales`** table → **`New measure`**.
4. Type the measure name on the left of `:=` and the DAX on the right, e.g.
   ```dax
   M_05_YTD := CALCULATE([Sales Amount Seed], DATESYTD('Date'[Date]))
   ```
5. Click ✅ to commit. Repeat for all 12 pillars.
6. **All measures must live on the `Sales` table** — the checker looks for them there.

> 💡 Tip: the seed measure **`[Sales Amount Seed] = SUM(Sales[Amount])`** is already there.
> Build every pillar by wrapping it in `CALCULATE(...)`.

---


## Step 1 — Connect to Cathedral_Model and sanity-check the data


In [ ]:
# Imports + bootstrap
import subprocess, sys, importlib
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "--disable-pip-version-check", "PyJWT>=2.6.0"],
               check=False, capture_output=True)
try:
    import sempy_labs as labs
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--disable-pip-version-check", "semantic-link-labs"],
                   check=True, capture_output=True)
    importlib.invalidate_caches()
    import sempy_labs as labs
import sempy.fabric as fabric

MODEL_NAME = "Cathedral_Model"
LAKEHOUSE  = "Cathedral_LH"

try:
    WORKSPACE_ID = fabric.get_notebook_workspace_id()
except Exception:
    WORKSPACE_ID = None
print(f"🏢 Workspace : {WORKSPACE_ID}")

# Sync the Lakehouse SQL endpoint (required after the Seed notebook ran).
try:
    from sempy_labs import refresh_sql_endpoint_metadata
    try:
        refresh_sql_endpoint_metadata(item=LAKEHOUSE, workspace=WORKSPACE_ID, type="Lakehouse")
    except TypeError:
        refresh_sql_endpoint_metadata(item=LAKEHOUSE, workspace=WORKSPACE_ID)
    print("✅ SQL endpoint synced")
except Exception as e:
    print(f"⚠️ SQL endpoint sync: {type(e).__name__}: {str(e)[:200]}")

# Force a Direct Lake reframe so the columns are queryable.
try:
    fabric.refresh_dataset(dataset=MODEL_NAME, workspace=WORKSPACE_ID, refresh_type="full")
    print("✅ Cathedral_Model refreshed")
except Exception as e:
    print(f"⚠️ refresh: {type(e).__name__}: {str(e)[:200]}")

# Smoke test: the base measure should return a positive number.
df = fabric.evaluate_dax(dataset=MODEL_NAME, workspace=WORKSPACE_ID,
                         dax_string="EVALUATE { [Sales Amount Seed] }")
print(f"📊 Sales Amount Seed = {df.iloc[0,0]:,.2f}")
print()
print("🟢 You are connected. Read the next section, then go build your 12 measures.")


## Step 2 — Read carefully: the 12 Pillars

For each pillar below, **create a measure in the `Sales` table** of `Cathedral_Model`
with the **exact name** shown (the checker is case-sensitive). The base measure
`[Sales Amount Seed] = SUM(Sales[Amount])` is already there — wrap it.

The checker will evaluate your measure in **3 filter contexts** and compare it to
the canonical result. The DAX you write also gets an **elegance score**: shorter +
less nesting = more points.

---

### 🪨 Pillar #1 — `M_01_Current` — Current Sales
Sum of sales in the current filter context. The warm-up.
- **Hint**: simply reference `[Sales Amount Seed]`.
- **Test contexts**: `'Date'[Year]=2024`  •  `'Date'[Year]=2025 & 'Date'[MonthNum]=6`  •  `'Date'[Year]=2024 & Customer[Region]="EU-North"`

### 🪨 Pillar #2 — `M_02_LastYear` — Sales Last Year
Same period one year before. Hint: `SAMEPERIODLASTYEAR('Date'[Date])`.

### 🪨 Pillar #3 — `M_03_YoY` — Sales YoY (absolute)
Current sales minus same period last year.

### 🪨 Pillar #4 — `M_04_YoYPct` — Sales YoY %
Percentage growth vs. last year. Hint: `DIVIDE`.

### 🪨 Pillar #5 — `M_05_YTD` — Sales Year-to-Date
Hint: `DATESYTD('Date'[Date])`.

### 🪨 Pillar #6 — `M_06_MTD` — Sales Month-to-Date
Hint: `DATESMTD('Date'[Date])`.

### 🪨 Pillar #7 — `M_07_QTD` — Sales Quarter-to-Date
Hint: `DATESQTD('Date'[Date])`.

### 🪨 Pillar #8 — `M_08_Rolling12` — Rolling 12-month Sales
Trailing 12 months. Hint: `DATESINPERIOD('Date'[Date], MAX('Date'[Date]), -12, MONTH)`.

### 🪨 Pillar #9 — `M_09_BestMonth` — Best Month Value
Highest monthly total inside the current filter. Hint: `MAXX(VALUES('Date'[MonthNum]), [Sales Amount Seed])`.

### 🪨 Pillar #10 — `M_10_PctOfYear` — % of Year
Share of the yearly total taken by the current month. Hint: `DIVIDE` with `ALL` over the date columns.

### 🪨 Pillar #11 — `M_11_AvgDailySales` — Average Daily Sales
Mean across visible days. Hint: `AVERAGEX(VALUES('Date'[Date]), [Sales Amount Seed])`.

### 🪨 Pillar #12 — `M_12_DistinctCustomers` — Distinct Customers
How many unique customers bought something. Hint: `DISTINCTCOUNT(Sales[CustomerKey])`.

---


## Step 3 — Build the measures in the web modeler

1. Switch to the workspace tab → open **`Cathedral_Model`** → **`Open data model`**.
2. Right-click the **`Sales`** table → **`New measure`**.
3. Use the **exact** measure name (e.g. `M_05_YTD`), enter the DAX, hit ✅.
4. Repeat for all 12.

> 🔁 **Reminder**: every measure must live on the **`Sales`** table.

## Step 4 — Grade your work

Open the **`CalcGroups_Check`** notebook in the workspace and run it.
It will:
- Verify each measure exists with the expected name.
- Run it in the 3 test contexts and compare to the canonical answer.
- Compute an **elegance score** (shorter + less nesting = better).
- Assign your **Architect rank** (Stonemason → Cathedral Builder).
- Log every attempt to `Cathedral_EH.CathedralEvents` (telemetry).

When all 12 pillars are 🟢, the Check notebook will unlock the **final challenge**.

🍀 *Good luck, architect.*
